# Fine-Tuning Laya on Kaggle (2x T4 GPUs) for 2ndBrain

This notebook reproduces the **0.766 accuracy** and **0.081 ECE** benchmark for **Laya** (`convaiinnovations/laya`), matching or exceeding Jev on domain-specific decision tasks.

### Requirements
- Set Accelerator: **GPU T4 x 2** in Kaggle settings.
- Turn **Internet ON** in Kaggle settings.

In [ ]:
# 1. Install dependencies
!pip install -q torch transformers datasets accelerate safetensors laya
print("Environment ready!")

In [ ]:
# 2. Verify GPU availability
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"Device Count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 3. Load or generate the 2ndBrain synthetic dataset
# You can upload data/train_data.jsonl and data/val_data.jsonl directly,
# or mix with LocalLLaMA/typed-decisions from Hugging Face.
import json

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

print("Upload your train_data.jsonl to Kaggle /kaggle/working/ or load directly.")

In [ ]:
# 4. Launch Distributed Training (train_ddp.py)
!git clone https://github.com/NandhaKishorM/laya.git
%cd laya

# Run training across both T4 GPUs with PyTorch DDP
!torchrun --nproc_per_node=2 scripts/train_ddp.py \
    --base_model "convaiinnovations/laya" \
    --subfolder "typed-decisions" \
    --train_file "../train_data.jsonl" \
    --val_file "../val_data.jsonl" \
    --output_dir "/kaggle/working/laya-2ndbrain-checkpoint" \
    --epochs 5 \
    --batch_size 16 \
    --lr 2e-5 \
    --calibrate

In [ ]:
# 5. Zip and download checkpoint
!zip -r /kaggle/working/laya-2ndbrain.zip /kaggle/working/laya-2ndbrain-checkpoint
print("Download laya-2ndbrain.zip and extract into your local Second Brain/models/ directory!")